<a href="https://colab.research.google.com/github/hrihaanG/Panvision/blob/segmentation/notebooks/colab_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PanVision — segmentation experiments (U-Net candidates, selection, test, export)

| Run | Stage | Model | Input | Role |
|---|---|---|---|---|
| P1 | pancreas | U-Net | 2D (1 slice) | baseline |
| P2 | pancreas | U-Net | 2.5D (3 slices) | candidate |
| P3 | pancreas | nnU-Net 2d | — | candidate, **trained in `colab_nnunet.ipynb`** |
| L0 | lesion | U-Net | 2.5D, full slice (no ROI) | no-cascade baseline |
| L1 | lesion | U-Net | 2.5D, pancreas ROI | baseline |
| L2 | lesion | Attention U-Net | 2.5D, pancreas ROI | candidate |
| L3 | lesion | best of L1/L2 | Dice + CE loss | loss ablation (L1/L2 use Dice + Focal) |

Selection uses **val** only. **Test** is run once at the end (section 6).

**Before the first run:** open the shared Drive link → right-click the folder → *Organize → Add shortcut* → *My Drive*.
Then **Runtime → Change runtime type → T4 GPU**.

Free Colab disconnects after a few hours. Nothing is lost: models, logs and results are saved to the shared Drive
folder. After a disconnect, run **Setup** again, then re-run the cell that was interrupted — training resumes.

## 1. Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
%cd /content
# Leave the repo folder first, so re-running this cell is safe.
from google.colab import drive
drive.mount('/content/drive')
# Private repo: clone with a GitHub token stored as a Colab secret named GITHUB_TOKEN (key icon in the left bar).
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
# Update in place instead of deleting the folder, so a training run in the background keeps working.
if os.path.exists('/content/Panvision/.git'):
    !git -C /content/Panvision fetch -q https://{token}@github.com/hrihaanG/Panvision.git segmentation && git -C /content/Panvision reset -q --hard FETCH_HEAD
else:
    !git clone -q -b segmentation https://{token}@github.com/hrihaanG/Panvision.git /content/Panvision
!git -C /content/Panvision remote set-url origin https://github.com/hrihaanG/Panvision.git  # drop the token from git config
%cd /content/Panvision
assert os.path.exists('src/segmentation'), 'Clone failed: check the GITHUB_TOKEN secret and its notebook access'
!pip -q install monai SimpleITK

In [ ]:
import os, glob
hits = sorted(set(p[:-len('/processed/full')] for p in glob.glob('/content/drive/MyDrive/**/processed/full', recursive=True)))
print('\n'.join(hits) or 'Not found: add the shared folder as a shortcut in My Drive (see top).')
# The shared folder name contains spaces, which break shell commands: use a space-free link to it.
if not os.path.exists('/content/data'):
    os.symlink(hits[0], '/content/data')  # change hits[0] if more than one folder is listed
DATA = '/content/data'
os.environ['PANVISION_DATA_DIR'] = DATA
CACHE = '/content/cache'
M = f'{DATA}/models/segmentation'
R = f'{DATA}/outputs/segmentation/results'
NNUNET_PRED = f'{DATA}/outputs/segmentation/nnunet/predictions'
print('Using', DATA)

In [ ]:
# Training runs as a background job, so a browser disconnect does not stop it. After reconnecting, run the setup
# cells and this one, then run the interrupted training cell again: it re-attaches to the running job (or, after a
# runtime reset, resumes from last.pt). Interrupting a cell only stops watching; stop('<run>') ends the job.
from src.segmentation.colab_jobs import follow, launch, stop

def start(run, flags):
    if not launch(run, f'python -m src.segmentation.train {flags} --run {run} --cache-dir {CACHE}'):
        raise RuntimeError(f'{run} did not finish; see its log above')


## 2. Data inventory (check once)

In [ ]:
from src.segmentation.data import find_studies
studies = find_studies()
labelled = studies[studies.processed_labels != '']
print(len(studies), 'completed studies,', len(labelled), 'with labels')
print(labelled.groupby(['split', 'label', 'annotation_tier']).size().unstack(fill_value=0))

## 3. Pancreas candidates (~1 h each on a T4)
If the browser disconnects, training keeps going: reconnect, run the two setup cells and the helper cell, then run the same cell again to see its progress.

In [ ]:
start('P1_unet_2d', f'--stage pancreas --model unet --context 1')

In [ ]:
start('P2_unet_25d', f'--stage pancreas --model unet --context 3')

### Compare pancreas candidates on val (P3 is included once `colab_nnunet.ipynb` has finished)

In [ ]:
import os
for run in ['P1_unet_2d', 'P2_unet_25d']:
    !python -m src.segmentation.evaluate pancreas --checkpoint {M}/{run}/best.pt --split val --max-studies 100
runs = ['P1_unet_2d', 'P2_unet_25d'] + (['P3_nnunet_2d'] if os.path.exists(f'{R}/P3_nnunet_2d_val_cases.csv') else [])
for i, a in enumerate(runs):
    for b in runs[i + 1:]:
        print(a, 'vs', b, end=': ')
        !python -m src.segmentation.evaluate compare {R}/{a}_val_cases.csv {R}/{b}_val_cases.csv --metric dice

In [ ]:
# Selection rule: highest mean val Dice. If nnU-Net is within ~0.02 Dice of the best U-Net, prefer the U-Net
# (simpler to ship in the dashboard). mean_diff > 0 means the first run is better; p_value is the Wilcoxon test.
PANCREAS_BEST = 'P2_unet_25d'   # <- edit: 'P1_unet_2d', 'P2_unet_25d' or 'P3_nnunet_2d'

def pancreas_flag(run=None):
    run = run or PANCREAS_BEST
    return f'--pancreas-predictions {NNUNET_PRED}' if run == 'P3_nnunet_2d' else f'--pancreas-checkpoint {M}/{run}/best.pt'
PANCREAS_FLAG = pancreas_flag()
print(PANCREAS_FLAG)

## 4. Lesion candidates
Trained on reference-pancreas ROIs (L0: full slices) with manual lesion masks only; non-PDAC studies are negatives.

In [ ]:
start('L1_unet_roi', f'--stage lesion --model unet --max-train 0')

In [ ]:
start('L2_attention_unet_roi', f'--stage lesion --model attention_unet --max-train 0')

In [ ]:
start('L0_unet_fullslice', f'--stage lesion --model unet --roi none --max-train 150')

### Compare on val: oracle ROI and end-to-end (with the selected pancreas model)

In [ ]:
for run in ['L1_unet_roi', 'L2_attention_unet_roi', 'L0_unet_fullslice']:
    !python -m src.segmentation.evaluate lesion --checkpoint {M}/{run}/best.pt $PANCREAS_FLAG --split val
for a, b in [('L2_attention_unet_roi', 'L1_unet_roi'), ('L1_unet_roi', 'L0_unet_fullslice')]:
    for metric in ['dice', 'recall']:
        print(a, 'vs', b, metric, end=': ')
        !python -m src.segmentation.evaluate compare {R}/{a}_val_cases.csv {R}/{b}_val_cases.csv --metric {metric} --mode end_to_end

In [ ]:
# Selection rule: higher mean end-to-end val Dice between L1 and L2; recall breaks ties.
# (L0 is not a selection candidate: it shows what the pancreas ROI cascade adds.)
LESION_BEST = 'L2_attention_unet_roi'   # <- edit: 'L1_unet_roi' or 'L2_attention_unet_roi'
LESION_MODEL = 'attention_unet' if LESION_BEST.startswith('L2') else 'unet'

### L3: loss ablation (same model as LESION_BEST, Dice + CE instead of Dice + Focal)

In [ ]:
L3 = f'L3_{LESION_MODEL}_roi_dicece'
start(L3, f'--stage lesion --model {LESION_MODEL} --loss dicece --max-train 0')
!python -m src.segmentation.evaluate lesion --checkpoint {M}/{L3}/best.pt $PANCREAS_FLAG --split val
for metric in ['dice', 'recall']:
    print(LESION_BEST, 'vs', L3, metric, end=': ')
    !python -m src.segmentation.evaluate compare {R}/{LESION_BEST}_val_cases.csv {R}/{L3}_val_cases.csv --metric {metric} --mode end_to_end

In [ ]:
LESION_FINAL = LESION_BEST   # <- set to L3 if it is better on val

## 5. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
groups = [['P1_unet_2d', 'P2_unet_25d'], ['L0_unet_fullslice', 'L1_unet_roi', 'L2_attention_unet_roi', L3]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, runs, title in zip(axes, groups, ['pancreas', 'lesion']):
    for run in runs:
        if os.path.exists(f'{M}/{run}/log.csv'):
            log = pd.read_csv(f'{M}/{run}/log.csv')
            ax.plot(log.step, log.val_dice, marker='o', label=run)
    ax.set(title=title, xlabel='step', ylabel='val Dice'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{R}/val_curves.png', dpi=150)

## 6. Final test evaluation — run once, after all choices above are fixed

In [ ]:
if PANCREAS_BEST == 'P3_nnunet_2d':
    !python -m src.segmentation.evaluate pancreas --predictions {NNUNET_PRED} --name P3_nnunet_2d --split test
else:
    !python -m src.segmentation.evaluate pancreas --checkpoint {M}/{PANCREAS_BEST}/best.pt --split test
!python -m src.segmentation.evaluate lesion --checkpoint {M}/{LESION_FINAL}/best.pt $PANCREAS_FLAG --split test --max-negatives 0

## 7. Export masks + measurements for teammates
Writes `outputs/segmentation/<study_id>/{pancreas,lesion}.nii.gz`, `roi.json` and `measurements.csv`.
(With nnU-Net selected, only val/test have nnU-Net pancreas masks.)

In [ ]:
!python -m src.segmentation.infer $PANCREAS_FLAG --lesion-checkpoint {M}/{LESION_FINAL}/best.pt --split val test